# Working

**Request:** [T08] Fee calculation for August

> New email from Sai Suram <sai@agents.agentstore.it.com>
> Subject: [T08] Fee calculation for August
> Thread ID: AAQkADI1N2Y5MTE3LTE1MDctNGY0Yy1iYzQ5LWEzNmE5NzAyYzk4NQAQAFQ7hOW67cVNmFjgerD81Q0=
> 
> ## THE RE

Each cell below is one run in the sandbox, in the order it happened, with whatever it printed and produced.


In [1]:
import pandas as pd

# Transaction data provided in the request
transactions_data = [
    ["Northwind", 42.00, "US", "no", "no"],
    ["Northwind", 3.50, "US", "no", "no"],
    ["Northwind", 128.00, "GB", "no", "no"],
    ["Northwind", 76.40, "US", "yes", "no"],
    ["Northwind", 250.00, "US", "no", "yes"],
    ["Contoso", 18.75, "US", "no", "no"],
    ["Contoso", 4.99, "DE", "no", "no"],
    ["Contoso", 310.00, "DE", "no", "no"],
    ["Contoso", 95.00, "US", "yes", "no"],
    ["Contoso", 62.30, "FR", "yes", "no"]
]

df = pd.DataFrame(transactions_data, columns=['Merchant', 'Amount', 'CardCountry', 'CardPresent', 'Refunded'])

# Monthly transaction counts (for volume tier)
monthly_counts = {'Northwind': 142, 'Contoso': 64}

# Function to calculate fee based on rules
def calculate_fee(row, merchant_total_transactions):
    amount = row['Amount']
    card_country = row['CardCountry']
    card_present = row['CardPresent']
    refunded = row['Refunded']

    # Rule 5: Micro-transactions
    if amount < 5.00:
        transaction_fee = 0.15
        percentage_component = 0  # No percentage for micro-transactions
        fixed_component = 0.15
    else:
        # Rule 4: Card-present
        if card_present == 'yes':
            percentage_rate = 0.019  # 1.9%
            fixed_component = 0.10
        else:
            # Rule 1: Standard rate
            percentage_rate = 0.029  # 2.9%
            fixed_component = 0.30
            
            # Rule 2: Volume tier (applies only if not card-present)
            if merchant_total_transactions > 100:
                percentage_rate = 0.025  # 2.5%

        # Rule 3: International
        if card_country != 'US':
            percentage_rate += 0.010 # Add 1.0 percentage point
        
        percentage_component = amount * percentage_rate
        transaction_fee = percentage_component + fixed_component
    
    # Rule 6: Refunds
    if refunded == 'yes':
        transaction_fee -= percentage_component # Percentage component is returned

    return transaction_fee

# Apply the fee calculation to each row
df['CalculatedFee'] = df.apply(lambda row: calculate_fee(row, monthly_counts[row['Merchant']]), axis=1)

# Calculate total fees per merchant
total_fees_per_merchant = df.groupby('Merchant')['CalculatedFee'].sum().reset_index()

# Prepare data for Excel output
# Per-transaction detail
per_transaction_df = df.copy()
per_transaction_df['Amount'] = per_transaction_df['Amount'].apply(lambda x: f'{x:.2f}')
per_transaction_df['CalculatedFee'] = per_transaction_df['CalculatedFee'].apply(lambda x: f'{x:.2f}')

# Summary of total fees
total_fees_summary_df = total_fees_per_merchant.copy()
total_fees_summary_df['CalculatedFee'] = total_fees_summary_df['CalculatedFee'].apply(lambda x: f'{x:.2f}')

# Output to a temporary Excel file
output_filename = 'August_Fee_Calculation.xlsx'
output_path = f'/tmp/output/{output_filename}'

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    total_fees_summary_df.to_excel(writer, sheet_name='Summary', index=False)
    per_transaction_df.to_excel(writer, sheet_name='Transaction Details', index=False)

print(f'File saved to {output_path}')

# Display data to confirm calculations
print('\nTotal Fees per Merchant:')
print(total_fees_per_merchant.to_json(orient='records'))
print('\nPer-Transaction Details:')
print(per_transaction_df.to_json(orient='records'))

File saved to /tmp/output/August_Fee_Calculation.xlsx

Total Fees per Merchant:
[{"Merchant":"Contoso","CalculatedFee":17.19545},{"Merchant":"Northwind","CalculatedFee":8.1316}]

Per-Transaction Details:
[{"Merchant":"Northwind","Amount":"42.00","CardCountry":"US","CardPresent":"no","Refunded":"no","CalculatedFee":"1.35"},{"Merchant":"Northwind","Amount":"3.50","CardCountry":"US","CardPresent":"no","Refunded":"no","CalculatedFee":"0.15"},{"Merchant":"Northwind","Amount":"128.00","CardCountry":"GB","CardPresent":"no","Refunded":"no","CalculatedFee":"4.78"},{"Merchant":"Northwind","Amount":"76.40","CardCountry":"US","CardPresent":"yes","Refunded":"no","CalculatedFee":"1.55"},{"Merchant":"Northwind","Amount":"250.00","CardCountry":"US","CardPresent":"no","Refunded":"yes","CalculatedFee":"0.30"},{"Merchant":"Contoso","Amount":"18.75","CardCountry":"US","CardPresent":"no","Refunded":"no","CalculatedFee":"0.84"},{"Merchant":"Contoso","Amount":"4.99","CardCountry":"DE","CardPresent":"no","Ref


[files written: August_Fee_Calculation.xlsx]
